# Getting Started with DNN 2

## Part 1 Binary classification problem

Two-class classification, or binary classification, may be the most widely applied kind of machine learning problem. Make sure you know the theoretical difference between single-label multi-class classification and multi-label classification. All of those problems use different model outputs and loss functions. If you are not sure about the difference - feel free to ask.


## The IMDB dataset

The aim of this lab is to classify movie reviews into "positive" reviews and "negative" reviews, just based on the text content of the reviews.
We'll be working with the "IMDB dataset", a set of 50,000 highly-polarized reviews from the Internet Movie Database. They are split into 25,000 reviews for training and 25,000 reviews for testing, each set consisting of 50% negative and 50% positive reviews. Just like the MNIST dataset, the IMDB dataset comes packaged with Keras. It has already been preprocessed: the reviews (sequences of words) have been turned into sequences of integers, where each integer stands for a specific word in a dictionary.

Please load the dataset from Keras. The argument num_words=10000 means that we will only keep the top 10,000 most frequently occurring words in the training data. Rare words will be discarded. This allows us to work with vector data of manageable size.

In [ ]:
from keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

Labels are saved as integers with numbers corresponding to the dictionary. We could use some word embeddings, but since we are only using simple fully-connected layer for binary classification to learn Keras, the best way would be one-hot encoding.

We will turn them into vectors of 0s and 1s. This would mean for instance turning the sequence  [3, 5] into a 10,000-dimensional vector that would be all-zeros except for indices 3 and 5, which would be ones. This might look unnecessary but is actually preferable for some neural networks.

In [ ]:
import numpy as np

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))  # Create an all-zero matrix of shape (len(sequences), dimension)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.  # set specific indices of results[i] to 1s
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

## Network architecture

Based on the previous ex. please implement a deep network architecture shown in the following Figure. You can also try your own network.

Think about the last layer. Which activation function should we use? How many neurons?

Which optimizer would you use? What about loss functions and metrics?

![model](...)


In [ ]:
#TODO Create the model

#TODO Compile the model

## Validation and Training

In order to monitor during training the accuracy of the model on data that it has never seen before, we will create a "validation set" by setting apart 10,000 samples from the original data (suggested split: 25000 train, 10000 validation, 15000 test). You can do it manually or use ''train_test_split'' from ''sklearn'' library.

Fit the model. The network should achieve at least acc: 0.9998 and val_acc: 0.86.

In [ ]:
#TODO: divide the dataset according to the description above

#TODO: Train the model

#model.fit(..., ..., ...,  validation_data=(x_val, y_val)))


## Visualisation and Testing


The `history = model.fit()` returns a History object. This object has a member history, which is a dictionary containing data about everything that happened during training. Let's take a look at it:

In [ ]:
history_dict = history.history
history_dict.keys()

It contains 4 entries: one per metric that was being monitored, during training and during validation. Please use Matplotlib to plot the training and validation loss side by side, as well as the training and validation accuracy.

In [ ]:
#TODO: plot

As you can see, the training loss decreases with every epoch, and the training accuracy increases with every epoch. In precise terms, what you are seeing is "overfitting": after the second epoch, we are over-optimizing on the training data, and we ended up learning representations that are specific to the training data and do not generalize to data outside of the training set.

Please check how that model performs on the test set too!!

In [ ]:
#TODO: evaluate the results on the test set

# Part 2 Hyperparameters

As we stated before, there is no ideal solution to each deep learning task. To decide how many layers to add or which activation functions use you need practice and usually a lot of trials and errors. Some people even say that implementing a neural network from scratch is more art than science.

# Parameter optimisation

Even then, after you decide what your model will look like, there is a matter of training, which also requires additional parameters. These numbers, that you already encountered, like a number of epochs, learning rate or optimizer are called hyperparameters. Training a neural network is basically an optimization problem, so you can think of setting those parameters as optimizing an optimization. It's not an easy task. There are, however, some tools that can help with that.

# Callbacks

One of the simplest methods helpful in parameter selection is using Callbacks. These are basic functions that let you act on the model during training. They allow you to interrupt training to save a model, load a different weight set, or otherwise alter the state of the model. Here are some examples of ways you can use callbacks:
  * Model checkpointing—saving the current weights of the model at different points during training.
  * Early stopping—interrupting training when the validation loss is no longer improving (and of course, saving the best model obtained during training).
  * Dynamically adjusting the value of certain parameters during training — such as the learning rate of the optimizer.
  * Logging training and validation metrics during training, or visualizing the representations learned by the model as they’re updated.

There are many default methods that you can explore in `keras.callbacks` such as `ModelCheckpoint`, `EarlyStopping` or `LearningRateScheduler`. However, one of the most useful things is that you can define your own callback and pass it to the model. So first, check if the desired functionality is available in Keras and if it isn't write it yourself.

To do so, you can use functions that trigger when certain action occurs:
  * `on_epoch_begin`
  * `on_epoch_end`
  * `on_batch_begin`
  * `on_batch_end`
  * `on_train_begin`
  * `on_train_end`

You can define function behavior after the trigger. In an example below, we check after each epoch if the training accuracy reaches a certain threshold (in this case 90%). If it does, we stop training. This is a useful method to decide on a number of epochs and avoid overfitting. You can read more about callbacks in [Keras Callbacks Documentation](https://keras.io/api/callbacks/)

First, you need to define a callback class, then create a callback instance and pass it as an argument when using `.fit()`. You can change parameters or callback behaviour as you wish, as long as it succeeds to stop training earlier.



In [ ]:

class myCallback(keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if(logs.get('accuracy')>0.9):
      print("\nReached 90% accuracy so cancelling training!")
      self.model.stop_training = True

callbacks = myCallback()

model.fit(x_train, y_train, epochs=10, callbacks=[callbacks])


## Optimisation algorithms

There are various libraries with tools that automatically set different parameters:
*   Scikit-Learn
*   Optuna
*   Keras itself

provides

One of the most popular optimisation alghorithms are GridSearch and RandomSearch. You can read more about them in the documentation below. More sophisticated methods, like genetic algorithms or particle swarm optimization, are sometimes used too.

Your last task this week is to use Grid Search or Random Search Optimisation algorithm to optimise one of the networks from previous tasks. Choose two parameters (for example number of epochs and batch size) and try two different values for each parameter. Grid Search method will therefore create a mesh of these parameters, with four elements (2x2). This process will take more time and computation power, as it will basically train the model four times and compare results (or more if you use cross-validation).

[Grid Search Method](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

[Random Search Method](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

Try using natvie Keras Tuner first. If you encountered some problems or if you want to test other libraries, try Sklearn or Optuna.

[Keras Tuner - Documentation](https://keras.io/keras_tuner/|)

[Keras Tuner - Tutorial](https://keras.io/guides/keras_tuner/getting_started/)

[Keras Tuner - training parameters](https://github.com/keras-team/keras-tuner/issues/122)

In [ ]:
#Wrap a model inside a function, with optimised parameters as arguments.


def build_model(hp):
    network = models.Sequential()
    network.add(#TO DO)

    network.compile(#TO DO)

    return network

#Set up a Random Search tuner with a model, objective function and max number of iterations.

tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=5)

tuner.search(train_images, train_labels, epochs=5, batch_size=128, validation_data=(val_images, val_labels))
best_model = tuner.get_best_models()[0]

# Sklear (optional)

In [ ]:
from sklearn.model_selection import GridSearchCV
from keras.wrappers.scikit_learn import KerasClassifier
# alternatively from scikeras.wrappers import KerasClassifier
# in Google Colab you may need to use !pip install scikeras
# in a block before for it to work properly.

def create_model():
    network = models.Sequential()
    network.add(#TO DO)

    network.compile(#TO DO)

    return network

model = KerasClassifier(build_fn=create_model)

par1_batch = [32, 64]
par2_epoch = [3, 5]

param_grid = dict(batch_size=par1_batch,
                  epochs=par2_epoch
                  )

grid = GridSearchCV(estimator=model,
                    param_grid=param_grid,
                    cv = 3
                    )

grid_results = grid.fit(X,Y)

print("Best: %f using %s" % (grid_results.best_score_, grid_results.best_params_))